# Deep Past Challenge - ByT5 Training v5

**LESSON FROM v4**: Fine-tuning with lr=5e-5 caused catastrophic forgetting.  
The base model scores 35.5 on LB, but after v4 fine-tuning it dropped to 28.83.

### v5 Strategy: Ultra-Conservative Fine-tuning
- **Baseline first**: Eval base model BEFORE training to establish a floor
- **Tiny LR**: 5e-7 (100x lower than v4!) — just nudge, don't overwrite
- **1 epoch only**: Single pass, stop before overfitting
- **No label smoothing**: Don't force the model to unlearn correct predictions
- **Highest-quality data only**: strategy_a (895 sentences) — the cleanest alignments
- **Abort if worse**: Skip Phase 2 if Phase 1 doesn't improve over baseline

### Input Datasets
1. `akkadian-processed-data` - train_split.csv and val_split.csv
2. `byt5-akkadian-mbr-v2` - mattiaangeli's pre-trained model (attach from Kaggle Models)

In [ ]:
!pip install -q sacrebleu transformers[torch] accelerate

import os, json, math, gc, time, shutil
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from torch.utils.data import Dataset
import sacrebleu

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f}GB")
    IS_H100 = 'H100' in gpu_name
    IS_A100 = 'A100' in gpu_name
else:
    IS_H100 = IS_A100 = False

# Check disk space
import subprocess
result = subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\nDisk space:\n{result.stdout}")

start_time = time.time()

In [ ]:
# ============================================================
# CONFIGURATION - v5 Ultra-conservative
# ============================================================

# Data paths
TRAIN_FILE = "/kaggle/input/akkadian-processed-data/train_split.csv"
VAL_FILE   = "/kaggle/input/akkadian-processed-data/val_split.csv"
if not Path(TRAIN_FILE).exists():
    TRAIN_FILE = "../data_processed/combined/train_split.csv"
    VAL_FILE   = "../data_processed/combined/val_split.csv"

# Model detection
MODEL_NAME = None
for p in [
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2",
]:
    if Path(p).exists():
        MODEL_NAME = p
        break

if MODEL_NAME is None:
    MODEL_NAME = "google/byt5-large"
    print("WARNING: Pre-trained Akkadian model not found!")
    print("         Falling back to google/byt5-large")
    FROM_PRETRAINED_AKKADIAN = False
else:
    FROM_PRETRAINED_AKKADIAN = True
    print(f"Using pre-trained Akkadian model: {MODEL_NAME}")

OUTPUT_DIR = Path("/kaggle/working/model") if Path("/kaggle").exists() else Path("../models/byt5-akkadian")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sequence lengths
MAX_SRC_LEN = 384
MAX_TGT_LEN = 384
PREFIX = "translate Akkadian to English: "
SEED = 42

# Batch size
if IS_H100 or IS_A100:
    BATCH_SIZE = 4
    GRAD_ACCUM = 8
else:
    BATCH_SIZE = 2
    GRAD_ACCUM = 16

USE_BF16 = IS_H100 or IS_A100
USE_FP16 = not USE_BF16

# ============================================================
# PHASE CONFIG — v5 ultra-conservative
# ============================================================
if FROM_PRETRAINED_AKKADIAN:
    # v4 FAILED with lr=5e-5 (catastrophic forgetting: 35.5 → 28.83)
    # v5: 100x lower LR, 1 epoch, no label smoothing, best data only
    PHASES = {
        1: {
            'epochs': 1,
            'lr': 5e-7,          # 100x lower than v4!
            'warmup': 50,
            'label_smoothing': 0.0,  # Don't unlearn correct predictions
            'desc': 'Ultra-conservative: strategy_a only (highest quality)',
            'sources': ['strategy_a'],  # Only 895 best-aligned sentences
        },
        2: {
            'epochs': 1,
            'lr': 2e-7,
            'warmup': 50,
            'label_smoothing': 0.0,
            'desc': 'Broader sentence data (if Phase 1 improved)',
            'sources': ['strategy_a', 'strategy_b_aligned', 'strategy_c'],
        },
    }
else:
    PHASES = {
        1: {'epochs': 5,  'lr': 3e-4, 'warmup': 300, 'label_smoothing': 0.1, 'desc': 'All data', 'sources': None},
        2: {'epochs': 10, 'lr': 5e-5, 'warmup': 200, 'label_smoothing': 0.1, 'desc': 'OA-only', 'sources': ['strategy_a','strategy_b_short','strategy_b_aligned','strategy_b_doc','strategy_c','lexicon']},
        3: {'epochs': 5,  'lr': 1e-5, 'warmup': 100, 'label_smoothing': 0.1, 'desc': 'Sentence-level', 'sources': ['strategy_a','strategy_b_aligned','strategy_c']},
    }

MAX_BYTE_LEN_FILTER = 512
NUM_WORKERS = 4

print(f"Max lengths: src={MAX_SRC_LEN}, tgt={MAX_TGT_LEN}")
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"Precision: {'bf16' if USE_BF16 else 'fp16'}")
print(f"Phases: {list(PHASES.keys())}")
print(f"Fine-tuning from Akkadian model: {FROM_PRETRAINED_AKKADIAN}")
if FROM_PRETRAINED_AKKADIAN:
    print(f"  Phase 1 LR: {PHASES[1]['lr']} (v4 was 5e-5 — 100x too high)")
    print(f"  Phase 1 Epochs: {PHASES[1]['epochs']}")
    print(f"  Label smoothing: {PHASES[1]['label_smoothing']}")

In [ ]:
# ============================================================
# PREPROCESSING + DATASET + HELPERS
# ============================================================

import re

_RE_BIG_GAP = re.compile(r'(\.{3,}|\u2026+|\u2026\u2026)')
_RE_SMALL_GAP = re.compile(r'(xx+|\s+x\s+)')

def minimal_preprocess(text):
    """Minimal preprocessing - only gap normalization."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = _RE_BIG_GAP.sub('<big_gap>', text)
    text = _RE_SMALL_GAP.sub('<gap>', text)
    return text.strip()


class PreTokenizedDataset(Dataset):
    """Pre-tokenized dataset cached in RAM."""
    def __init__(self, df, tokenizer, max_src=384, max_tgt=384, prefix="", preprocess_fn=None):
        self.samples = []
        print(f"  Pre-tokenizing {len(df)} samples...", end=" ", flush=True)
        t0 = time.time()
        for _, row in df.iterrows():
            src_text = str(row['transliteration'])
            if preprocess_fn:
                src_text = preprocess_fn(src_text)
            src_text = prefix + src_text
            tgt_text = str(row['translation'])
            src = tokenizer(src_text, max_length=max_src, truncation=True)
            tgt = tokenizer(text_target=tgt_text, max_length=max_tgt, truncation=True)
            self.samples.append({
                'input_ids': src['input_ids'],
                'attention_mask': src['attention_mask'],
                'labels': tgt['input_ids'],
            })
        print(f"done in {time.time()-t0:.1f}s")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def filter_for_phase(data, phase_cfg):
    """Filter training data based on phase config."""
    sources = phase_cfg.get('sources')
    if sources is None:
        return data  # Use all data
    filtered = data[data['source'].isin(sources)]
    print(f"  Filtered to {len(filtered)} rows from sources: {sources}")
    return filtered


def filter_by_length(df, max_bytes=512):
    src_len = df['transliteration'].astype(str).apply(lambda x: len(x.encode('utf-8')))
    tgt_len = df['translation'].astype(str).apply(lambda x: len(x.encode('utf-8')))
    mask = (src_len <= max_bytes) & (tgt_len <= max_bytes)
    filtered = df[mask]
    dropped = len(df) - len(filtered)
    if dropped > 0:
        print(f"  Filtered out {dropped} samples > {max_bytes} bytes ({dropped/len(df)*100:.1f}%)")
    return filtered


def cleanup_dir(path):
    path = Path(path)
    if path.exists():
        size_mb = sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / 1e6
        shutil.rmtree(path)
        print(f"  Cleaned up {path.name}: freed {size_mb:.0f} MB")


print("Functions defined.")

In [ ]:
# ============================================================
# LOAD DATA & MODEL
# ============================================================

train_data = pd.read_csv(TRAIN_FILE).dropna(subset=['transliteration','translation'])
val_data = pd.read_csv(VAL_FILE).dropna(subset=['transliteration','translation'])
print(f"Raw: {len(train_data)} train | {len(val_data)} val")
print(train_data['source'].value_counts())

train_data = filter_by_length(train_data, MAX_BYTE_LEN_FILTER)
print(f"After filtering: {len(train_data)} train")

# Load model
print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16 if USE_BF16 else torch.float32,  # Fixed: dtype not torch_dtype
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.0f}M")

# Preprocessing
preprocess_fn = minimal_preprocess if FROM_PRETRAINED_AKKADIAN else None
print(f"Preprocessing: {'minimal (gap norm only)' if preprocess_fn else 'none'}")

# Pre-tokenize validation set
print("\nPre-tokenizing validation set...")
val_ds = PreTokenizedDataset(val_data, tokenizer, MAX_SRC_LEN, MAX_TGT_LEN, PREFIX, preprocess_fn=preprocess_fn)
print(f"Elapsed: {(time.time()-start_time)/60:.1f} min")

In [ ]:
# ============================================================
# TRAINING with BASELINE EVAL + ABORT-IF-WORSE
# ============================================================
# v5 KEY INSIGHT: Always know how the base model performs BEFORE training.
# If fine-tuning makes it worse, ABORT and save the untouched model.

import copy

all_results = {}
data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    padding=True,
    label_pad_token_id=-100,
)

def generate_eval(model, tokenizer, val_ds, max_samples=50, num_beams=5):
    """Run generation-based evaluation."""
    model.eval()
    device = next(model.parameters()).device
    preds_list, labels_list = [], []
    n = min(max_samples, len(val_ds))
    print(f"  Evaluating on {n} samples (beams={num_beams})...", flush=True)
    t0 = time.time()
    
    for i in range(n):
        sample = val_ds[i]
        input_ids = torch.tensor([sample['input_ids']], device=device)
        attention_mask = torch.tensor([sample['attention_mask']], device=device)
        with torch.no_grad():
            out = model.generate(
                input_ids=input_ids, attention_mask=attention_mask,
                max_new_tokens=256, num_beams=num_beams,
                length_penalty=1.3, repetition_penalty=1.2,
            )
        pred_text = tokenizer.decode(out[0], skip_special_tokens=True).strip()
        label_ids = [t for t in sample['labels'] if t != -100 and t != tokenizer.pad_token_id]
        label_ids_clipped = np.clip(label_ids, 0, 258).tolist()
        label_text = tokenizer.decode(label_ids_clipped, skip_special_tokens=True).strip()
        preds_list.append(pred_text)
        labels_list.append(label_text)
    
    bleu = sacrebleu.corpus_bleu(preds_list, [labels_list]).score
    chrf = sacrebleu.corpus_chrf(preds_list, [labels_list], word_order=2).score
    geo = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0
    
    print(f"  Done in {time.time()-t0:.1f}s")
    print(f"  BLEU={bleu:.2f} | chrF++={chrf:.2f} | GeoMean={geo:.2f}")
    for j in range(min(3, n)):
        print(f"    [{j}] pred: {preds_list[j][:80]}")
        print(f"    [{j}]  ref: {labels_list[j][:80]}")
    
    return {'bleu': round(bleu, 2), 'chrf_pp': round(chrf, 2), 'geo_mean': round(geo, 2)}


# ============================================================
# STEP 1: BASELINE EVAL (before any training)
# ============================================================
print("=" * 60)
print("BASELINE: Evaluating pre-trained model BEFORE any training")
print("=" * 60)
baseline = generate_eval(model, tokenizer, val_ds, max_samples=80, num_beams=5)
all_results['baseline'] = baseline
baseline_geo = baseline['geo_mean']
print(f"\n>>> BASELINE GeoMean = {baseline_geo}")
print(f"    (Must beat this, otherwise fine-tuning is harmful)")

# Save a copy of base model weights (to restore if fine-tuning hurts)
base_state = copy.deepcopy(model.state_dict())
best_geo = baseline_geo
best_phase = 'baseline'

# ============================================================
# STEP 2: ULTRA-CONSERVATIVE FINE-TUNING
# ============================================================
for phase_num in sorted(PHASES.keys()):
    cfg = PHASES[phase_num]
    phase_train_df = filter_for_phase(train_data, cfg)
    
    print(f"\n{'='*60}")
    print(f"PHASE {phase_num}: {cfg['desc']}")
    print(f"  Data: {len(phase_train_df)} | Epochs: {cfg['epochs']} | LR: {cfg['lr']}")
    print(f"  Label smoothing: {cfg['label_smoothing']}")
    print(f"  Current best: {best_geo:.2f} (from {best_phase})")
    print(f"{'='*60}")
    
    if len(phase_train_df) == 0:
        print("Skipping -- no data")
        continue
    
    # Pre-tokenize
    train_ds = PreTokenizedDataset(
        phase_train_df, tokenizer, MAX_SRC_LEN, MAX_TGT_LEN, PREFIX,
        preprocess_fn=preprocess_fn,
    )
    
    phase_dir = OUTPUT_DIR / f"phase{phase_num}"
    steps_per_epoch = max(1, len(phase_train_df) // (BATCH_SIZE * GRAD_ACCUM))
    log_steps = max(10, steps_per_epoch // 5)
    
    args = Seq2SeqTrainingArguments(
        output_dir=str(phase_dir),
        num_train_epochs=cfg['epochs'],
        learning_rate=cfg['lr'],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=cfg['warmup'],
        weight_decay=0.01,
        label_smoothing_factor=cfg['label_smoothing'],
        bf16=USE_BF16, fp16=USE_FP16,
        predict_with_generate=False,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=log_steps,
        report_to='none',
        seed=SEED,
        dataloader_num_workers=NUM_WORKERS,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
    )
    
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
    )
    
    trainer.train()
    
    # Evaluate after this phase
    print(f"\nPhase {phase_num} done. Evaluating...")
    gen_results = generate_eval(model, tokenizer, val_ds, max_samples=80, num_beams=5)
    all_results[f'phase{phase_num}'] = gen_results
    phase_geo = gen_results['geo_mean']
    
    print(f"\n>>> Phase {phase_num}: GeoMean={phase_geo:.2f} (baseline was {baseline_geo:.2f})")
    
    if phase_geo > best_geo:
        print(f"    ✅ IMPROVED! ({best_geo:.2f} → {phase_geo:.2f})")
        best_geo = phase_geo
        best_phase = f'phase{phase_num}'
        # Save this as our new best
        base_state = copy.deepcopy(model.state_dict())
    else:
        print(f"    ❌ WORSE or NO IMPROVEMENT ({best_geo:.2f} → {phase_geo:.2f})")
        print(f"    Restoring best model (from {best_phase})")
        model.load_state_dict(base_state)
        
        if FROM_PRETRAINED_AKKADIAN:
            print(f"    ABORTING further phases — fine-tuning is not helping")
            break
    
    del trainer, train_ds
    gc.collect()
    torch.cuda.empty_cache()
    cleanup_dir(phase_dir)

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE — Best: {best_phase} (GeoMean={best_geo:.2f})")
print(f"Total time: {(time.time()-start_time)/60:.1f} min")
print(f"{'='*60}")

In [ ]:
# ============================================================
# SAVE BEST MODEL (might be untouched base if fine-tuning hurt)
# ============================================================

final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

# The model already has best weights loaded (either base or fine-tuned)
model.save_pretrained(str(final_dir), safe_serialization=True)
tokenizer.save_pretrained(str(final_dir))

with open(str(final_dir / 'training_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"Model saved to: {final_dir}")
print(f"Best model: {best_phase} (GeoMean={best_geo:.2f})")
for phase, r in all_results.items():
    marker = " ← BEST" if phase == best_phase else ""
    geo = r.get('geo_mean', '?')
    bleu = r.get('bleu', '?')
    chrf = r.get('chrf_pp', '?')
    print(f"  {phase}: GeoMean={geo} | BLEU={bleu} | chrF++={chrf}{marker}")

model_size = sum(f.stat().st_size for f in final_dir.rglob('*') if f.is_file()) / 1e9
print(f"\nModel size: {model_size:.2f} GB")
print(f"Total time: {(time.time()-start_time)/60:.1f} min")

result = subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\nDisk space:\n{result.stdout}")

In [ ]:
# ============================================================
# QUICK TEST — with real Akkadian diacritics
# ============================================================

model.eval()
device = next(model.parameters()).device

# Real examples with proper diacritics (like training data)
test_inputs = [
    "um-ma ša-lim-a-šùr-ma a-na šu-be-lim-ma",
    "1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé",
    "KIŠIB ma-nu-ba-lúm-a-šùr DUMU ṣí-lá-{d}IM",
    "a-na a-lim{ki} i-lá-ak-ma",
]

print("Sample translations:\n")
for text in test_inputs:
    enc = tokenizer(
        PREFIX + text, max_length=MAX_SRC_LEN,
        truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=256,
            num_beams=5, length_penalty=1.0,
        )
    translation = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"  AKK: {text}")
    print(f"  ENG: {translation}")
    print()